In [ ]:
# install in env
#%pip install great_expectations

In [4]:
import uuid
from typing import Tuple
from pyspark.sql import DataFrame, functions as F
import great_expectations as gx
from great_expectations import expectations as gxe
from functools import reduce

StatementMeta(, b9d5b651-1e2e-4bac-910c-46e07d21ad42, 15, Finished, Available, Finished, False)

# Init context

In [ ]:
context = gx.get_context()

In [ ]:
#Load rules

In [ ]:
def load_rules(dataset_name):

    rules_df = spark.sql(f"""
        SELECT dataset_name,
               rule_id,
               dimension,
               rule_type,
               column_name,
               expectation,
               expectation_kwargs,
               severity,
               active
        FROM dq_rules
        WHERE dataset_name = '{dataset_name}' AND active = true
    """)

    return [row.asDict() for row in rules_df.collect()]

In [ ]:
def get_batch_from_dataframe(df: DataFrame, dataset_name: str):

    data_source_name = f"{dataset_name}_spark_source"
    data_asset_name = f"{dataset_name}_data_asset"
    batch_definition_name = f"{dataset_name}_batch"

    try:
        data_source = context.data_sources.get(data_source_name)
    except LookupError:
        data_source = context.data_sources.add_spark(name=data_source_name)

    try:
        data_asset = data_source.get_asset(data_asset_name)
    except LookupError:
        data_asset = data_source.add_dataframe_asset(name=data_asset_name)

    try:
        batch_definition = data_asset.get_batch_definition(batch_definition_name)
    except LookupError:
        batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)

    batch_parameters = {"dataframe": df}
    batch = batch_definition.get_batch(batch_parameters=batch_parameters)
    return batch

# Apply rules

In [ ]:
def apply_rules_to_batch(batch, rules, pk_column):

    validation_results = []

    complete_result_format = {
        "result_format": "COMPLETE",
        "include_unexpected_rows": True,
        "unexpected_index_column_names": [pk_column]
    }

    for rule in rules:

        col = rule['column_name']
        expectation_name = rule['expectation']
        expression = rule.get('expectation_kwargs')
        severity = map_severity(rule.get('severity'))

        if expectation_name == "expect_not_null":
            expectation = gxe.ExpectColumnValuesToNotBeNull(
                column=col,
                severity=severity
            )

        elif expectation_name == "expect_unique":
            expectation = gxe.ExpectColumnValuesToBeUnique(
                column=col,
                severity=severity
            )

        elif expectation_name == "expect_regex":
            expectation = gxe.ExpectColumnValuesToMatchRegex(
                column=col,
                regex=expression,
                severity=severity
            )

        elif expectation_name in ("expect_fk", "expect_reference"):
            value_set = get_reference_values(expression)
            expectation = gxe.ExpectColumnValuesToBeInSet(
                column=col,
                value_set=value_set,
                severity=severity
            )

        elif expectation_name == "expect_custom_sql":
            sql_query = expression

            # Debug prints
            print(f"\n=== Running Custom SQL Rule: {rule['rule_id']} ===")
            print(f"Column: {col}")
            print(f"SQL being executed:\n{sql_query}\n")

            expectation = gx.expectations.UnexpectedRowsExpectation(
                unexpected_rows_query=sql_query,
                description=f"Custom SQL rule {rule['rule_id']} failed",
                meta={"rule_id": rule['rule_id'], "column": col}
            )

        else:
            continue

        # Validate batch
        result = batch.validate(
            expectation,
            result_format=complete_result_format
        )

        # custom sql output
        if expectation_name == "expect_custom_sql":
            unexpected_rows = result['result'].get('details', {}).get('unexpected_rows', [])
            failed_rows = len(unexpected_rows)
            result['success'] = failed_rows == 0
            result['result']['unexpected_count'] = failed_rows

            print(f"Custom SQL rule {rule['rule_id']} detected {failed_rows} failing rows.")
            if failed_rows:
                print("Sample failing row:", unexpected_rows[0])

        validation_results.append(result)

    return validation_results


def summarize_validation_results(rules, validation_results):
    """
    Build a human-readable summary from GX validation results,
    including custom SQL unexpected rows.
    """

    summary = []
    for rule, result in zip(rules, validation_results):
        expectation_name = rule['expectation']
        col = rule['column_name']

        failed_rows = result['result'].get('unexpected_count', 0)

        summary.append({
            "rule": rule['rule_id'],
            "column": col if expectation_name != "expect_custom_sql" else "N/A",
            "success": result['success'],
            "failed_rows": failed_rows
        })

    # Print summary
    print("\n=== Data Quality Validation Summary ===")
    for row in summary:
        print(f"Rule: {row['rule']} | Column: {row['column']} | Success: {row['success']} | Failed rows: {row['failed_rows']}")

    return summary

# Old apply rules (frozen)

In [ ]:
def apply_rules_to_batch(batch, rules, pk_column):
    """
    Apply all dq_rules expectations to a GX batch.
    """

    validation_results = []

    complete_result_format = {
        "result_format": "COMPLETE",
        "include_unexpected_rows": True,
        "unexpected_index_column_names": [pk_column]
    }

    for rule in rules:

        col = rule['column_name']
        expectation_name = rule['expectation']
        expression = rule.get('expectation_kwargs')
        severity = map_severity(rule.get('severity'))

        if expectation_name == "expect_not_null":
            expectation = gxe.ExpectColumnValuesToNotBeNull(
                column=col,
                severity=severity
            )

        elif expectation_name == "expect_unique":
            expectation = gxe.ExpectColumnValuesToBeUnique(
                column=col,
                severity=severity
            )

        elif expectation_name == "expect_regex":
            expectation = gxe.ExpectColumnValuesToMatchRegex(
                column=col,
                regex=expression,
                severity=severity
            )

        elif expectation_name in ("expect_fk", "expect_reference"):

            value_set = get_reference_values(expression)

            expectation = gxe.ExpectColumnValuesToBeInSet(
                column=col,
                value_set=value_set,
                severity=severity
            )

        else:
            continue

        result = batch.validate(
            expectation,
            result_format=complete_result_format
        )

        validation_results.append(result)

    return validation_results

# Log

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
import uuid

def write_failed_and_log(df, passed_df, failed_df, dataset_name, pk_column):

    run_id = str(uuid.uuid4())

    if not failed_df.rdd.isEmpty():
        failed_prepped = (
            failed_df
            .withColumn("run_id", F.lit(run_id))
            .withColumn("dataset_name", F.lit(dataset_name))
            .withColumn("record_data", F.to_json(F.struct("*")))
            .withColumn("pk", F.col(pk_column).cast("string"))
        )

        spark.sql("""
            CREATE TABLE IF NOT EXISTS dq_failed_rows (
                pk STRING,
                run_id STRING,
                dataset_name STRING,
                record_data STRING
            )
            USING DELTA
        """)

        delta_failed = DeltaTable.forName(spark, "dq_failed_rows")

        delta_failed.alias("tgt").merge(
            failed_prepped.alias("src"),
            "tgt.pk = src.pk AND tgt.run_id = src.run_id"
        ).whenNotMatchedInsert(values={
            "pk": "src.pk",
            "run_id": "src.run_id",
            "dataset_name": "src.dataset_name",
            "record_data": "src.record_data"
        }).whenMatchedUpdate(set={
            "record_data": "src.record_data"
        }).execute()

    spark.sql("""
        CREATE TABLE IF NOT EXISTS dq_run_log (
            run_id STRING,
            dataset_name STRING,
            run_time TIMESTAMP,
            total_rows BIGINT,
            passed_rows BIGINT,
            failed_rows BIGINT
        )
        USING DELTA
    """)

    total_rows = df.count()
    passed_rows = passed_df.count()
    failed_rows = failed_df.count()

    spark.sql(f"""
        INSERT INTO dq_run_log VALUES(
            '{run_id}',
            '{dataset_name}',
            current_timestamp(),
            {total_rows},
            {passed_rows},
            {failed_rows}
        )
    """)

    print(f"DQ run logged successfully: {run_id}")

# Validation

In [ ]:
from typing import Tuple
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def run_gx_validation(df: DataFrame, dataset_name: str, pk_column: str) -> Tuple[DataFrame, DataFrame]:

    # Load rules
    rules = load_rules(dataset_name)

    # Build GX batch
    batch = get_batch_from_dataframe(df, dataset_name)

    # Apply expectations
    validation_results = apply_rules_to_batch(batch, rules, pk_column)

    # Extract failed PKs
    failed_pks = extract_failed_pk_values(validation_results, pk_column)

    # Split dataframe
    passed_df, failed_df = split_passed_failed(df, failed_pks, pk_column)

    print(f"\nData Quality Validation Summary for dataset '{dataset_name}':")
    print(f"Total rows: {df.count()}")
    print(f"Passed rows: {passed_df.count()}")
    print(f"Failed rows: {failed_df.count()}\n")

    print("Rule Results:")

    for result in validation_results:

        expectation = result.expectation_config.type
        column = result.expectation_config.kwargs.get("column", "N/A")

        success = result.success
        unexpected_count = result.result.get("unexpected_count", 0)

        print(
            f"Rule: {expectation} | Column: {column} | "
            f"Success: {success} | Failed rows: {unexpected_count}"
        )

    return passed_df, failed_df

# Helper functions

In [ ]:
def map_severity(sev: str) -> str:

    if not sev:
        return "critical"
    sev_lower = sev.lower()
    if sev_lower in ("error", "critical"):
        return "critical"
    elif sev_lower == "warning":
        return "warning"
    elif sev_lower == "info":
        return "info"
    else:
        return "critical"

In [ ]:
def get_reference_values(ref_table_name: str) -> list:

    if not ref_table_name:
        return []
    
    try:
        df_ref = spark.table(f"dq_lh.dbo.{ref_table_name}")
        values = [row[0] for row in df_ref.collect()]
        return values
    except Exception as e:
        print(f"Warning: Could not load reference table '{ref_table_name}': {e}")
        return []

In [ ]:
def extract_failed_pk_values(validation_results, pk_column):

    failed_pks = set()

    for result in validation_results:

        # Custom SQL unexpected_rows,return dicts
        unexpected_rows = result.result.get("details", {}).get("unexpected_rows", [])
        for row in unexpected_rows:
            if isinstance(row, dict) and pk_column in row:
                failed_pks.add(row[pk_column])

        # Column-based unexpected_index_list
        unexpected_index_list = result.result.get("unexpected_index_list", [])
        for val in unexpected_index_list:
            if isinstance(val, dict) and pk_column in val:
                failed_pks.add(val[pk_column])
            elif isinstance(val, (int, str)):
                failed_pks.add(val)
            # ignore anything else

        #details.unexpected_index_list
        unexpected_index_details = result.result.get("details", {}).get("unexpected_index_list", [])
        for val in unexpected_index_details:
            if isinstance(val, dict) and pk_column in val:
                failed_pks.add(val[pk_column])
            elif isinstance(val, (int, str)):
                failed_pks.add(val)

    return failed_pks

In [ ]:
def split_passed_failed(df, failed_pks, pk_column):

    if not failed_pks:
        return df, df.limit(0)

    # Determine column type
    col_type = [f.dataType for f in df.schema.fields if f.name == pk_column][0]

    # Convert failed_pks to match column type
    if "IntegerType" in str(col_type):
        failed_pks_casted = [int(pk) for pk in failed_pks]
    else:
        failed_pks_casted = [str(pk) for pk in failed_pks]

    failed_df = df.filter(F.col(pk_column).isin(failed_pks_casted))
    passed_df = df.filter(~F.col(pk_column).isin(failed_pks_casted))

    return passed_df, failed_df

# Schema

In [ ]:
def load_schema_rules(dataset_name: str):
    """
    Load active schema rules from dq_rules table.
    """
    rules_df = spark.sql(f"""
        SELECT dataset_name,
               rule_id,
               column_name,
               expectation,
               expectation_kwargs,
               severity
        FROM dq_rules
        WHERE dataset_name = '{dataset_name}' AND active = true AND rule_type = 'schema'
    """)
    return [row.asDict() for row in rules_df.collect()]

In [ ]:
from pyspark.sql import DataFrame

def get_batch_from_dataframe(df: DataFrame, dataset_name: str):
    """
    Create a Great Expectations batch for a Spark DataFrame.
    """
    data_source_name = f"{dataset_name}_spark_source"
    data_asset_name = f"{dataset_name}_data_asset"
    batch_definition_name = f"{dataset_name}_batch"

    try:
        data_source = context.data_sources.get(data_source_name)
    except LookupError:
        data_source = context.data_sources.add_spark(name=data_source_name)

    try:
        data_asset = data_source.get_asset(data_asset_name)
    except LookupError:
        data_asset = data_source.add_dataframe_asset(name=data_asset_name)

    try:
        batch_definition = data_asset.get_batch_definition(batch_definition_name)
    except LookupError:
        batch_definition = data_asset.add_batch_definition_whole_dataframe(batch_definition_name)

    batch_parameters = {"dataframe": df}
    batch = batch_definition.get_batch(batch_parameters=batch_parameters)
    return batch

In [ ]:
import great_expectations.expectations.core as gxe
import json

def apply_schema_rules_to_batch(batch, rules):
    """
    Apply only R0 (schema/order) and R8 (column types) expectations to a GE Spark batch.
    Returns a list of validation results with per-column details.
    """
    validation_results = []

    for rule in rules:
        expectation_name = rule["expectation"]
        kwargs = rule.get("expectation_kwargs", "")

        # Table / schema column order check (R0)
        if expectation_name in ("expect_table_columns", "expect_schema"):
            expected_cols = [c["name"] for c in json.loads(kwargs)["columns"]]
            expectation = gxe.ExpectTableColumnsToMatchOrderedList(column_list=expected_cols)
            result = batch.validate(expectation)
            validation_results.append({
                "expectation": expectation.expectation_type,
                "column": "TABLE",
                "success": result.success,
                "details": result.result
            })

        # Multi-column type check (R8)
        elif expectation_name == "expect_column_types":
            schema_cols = json.loads(kwargs)["columns"]
            for col in schema_cols:
                expectation = gxe.ExpectColumnValuesToBeOfType(
                    column=col["name"],
                    type_=col["type"]
                )
                result = batch.validate(expectation)
                validation_results.append({
                    "expectation": expectation.expectation_type,
                    "column": col["name"],
                    "success": result.success,
                    "details": result.result
                })

        else:
            continue

    return validation_results

In [ ]:
def run_schema_validation_gx(df, dataset_name):
    """
    Run schema validation for R0 + R8 rules.
    Returns original DF, overall success, and validation results.
    """
    rules = load_schema_rules(dataset_name)
    if not rules:
        print("No schema rules found for dataset:", dataset_name)
        return df, True, []

    # Wrap Spark DF in GE batch
    batch = get_batch_from_dataframe(df, dataset_name)

    # Apply schema rules
    validation_results = apply_schema_rules_to_batch(batch, rules)

    # Overall success = all validations passed
    success = all(r["success"] for r in validation_results)

    # Print summary
    print(f"\nSchema Validation Summary for dataset '{dataset_name}':")
    for r in validation_results:
        print(
            f"Rule: {r['expectation']} | Column: {r['column']} | "
            f"Success: {r['success']} | Details: {r['details']}"
        )

    return df, success, validation_results

In [ ]:
from datetime import datetime
import uuid

def log_schema_result(dataset_name, df, success, validation_results):
    """
    Log schema validation results to dq_schema_log, including per-column errors from GE.
    Fail notebook if schema validation fails.
    """
    import json

    run_id = str(uuid.uuid4())
    run_time = datetime.now()

    # Get expected schema from R0
    rules = load_schema_rules(dataset_name)
    expected_rule = next((r for r in rules if r["expectation"] in ("expect_schema","expect_table_columns")), None)
    expected_schema = json.loads(expected_rule["expectation_kwargs"]) if expected_rule else {"columns": []}

    # Actual schema from DataFrame
    actual_schema = {"columns": []}
    for f in df.schema.fields:
        type_str = f.dataType.simpleString().capitalize() + "Type"
        actual_schema["columns"].append({"name": f.name, "type": type_str})

    # Build error message from GE details
    if validation_results:
        error_message = json.dumps(validation_results)
    else:
        error_message = None

    # Create Delta table if not exists
    spark.sql("""
    CREATE TABLE IF NOT EXISTS dq_schema_log (
        run_id STRING,
        dataset_name STRING,
        run_time TIMESTAMP,
        schema_success BOOLEAN,
        expected_schema STRING,
        actual_schema STRING,
        error_message STRING
    )
    """)

    # Write log
    log_row = [
        (
            run_id,
            dataset_name,
            run_time,
            success,
            json.dumps(expected_schema),
            json.dumps(actual_schema),
            error_message
        )
    ]

    columns = [
        "run_id",
        "dataset_name",
        "run_time",
        "schema_success",
        "expected_schema",
        "actual_schema",
        "error_message"
    ]

    log_df = spark.createDataFrame(log_row, schema=columns)
    log_df.write.mode("append").saveAsTable("dq_schema_log")

    print(f"\nSchema validation logged with run_id: {run_id}")

    # Fail notebook 
    if not success:
        raise Exception(
            f"Schema validation FAILED for dataset '{dataset_name}'. "
            f"Run ID: {run_id}. See dq_schema_log for details."
        )

    return run_id

In [ ]:
import great_expectations.expectations.core as gxe
import great_expectations as gx
import json

def apply_schema_and_custom_rules(batch, rules, pk_column=None):
 
    validation_results = []

    # Format for detailed GE results
    complete_result_format = {
        "result_format": "COMPLETE",
        "include_unexpected_rows": True,
    }
    if pk_column:
        complete_result_format["unexpected_index_column_names"] = [pk_column]

    for rule in rules:
        expectation_name = rule["expectation"]
        kwargs = rule.get("expectation_kwargs", "")

        # (R0)
        if expectation_name in ("expect_table_columns", "expect_schema"):
            expected_cols = [c["name"] for c in json.loads(kwargs)["columns"]]
            expectation = gxe.ExpectTableColumnsToMatchOrderedList(column_list=expected_cols)
            result = batch.validate(expectation)
            validation_results.append({
                "rule": expectation.expectation_type,
                "column": "TABLE",
                "success": result.success,
                "failed_rows": len(result.result.get("unexpected_index_list", []))
            })

        #  (R8)
        elif expectation_name == "expect_column_types":
            schema_cols = json.loads(kwargs)["columns"]
            for col in schema_cols:
                expectation = gxe.ExpectColumnValuesToBeOfType(
                    column=col["name"],
                    type_=col["type"]
                )
                result = batch.validate(expectation)
                validation_results.append({
                    "rule": expectation.expectation_type,
                    "column": col["name"],
                    "success": result.success,
                    "failed_rows": len(result.result.get("unexpected_index_list", []))
                })

        else:
            continue

    # Print a summary
    print(f"\nData Quality Validation Summary for dataset '{getattr(batch, 'name', 'batch')}':")
    print(f"Total rows: {batch.count()}")
    total_failed = sum(r['failed_rows'] for r in validation_results)
    print(f"Failed rows: {total_failed}")
    print("\nRule Results:")
    for r in validation_results:
        col = r['column'] if r['column'] else "N/A"
        print(f"Rule: {r['rule']} | Column: {col} | Success: {r['success']} | Failed rows: {r['failed_rows']}")

    return validation_results